# Minimal BYM2 Prototype in PyMC

This notebook presents a minimal prototype implementation of a BYM2-style spatial model in PyMC. 
It is intended to demonstrate feasibility, modeling understanding, and early validation of the 
structured + unstructured decomposition used in areal spatial models.

The notebook demonstrates the following key aspects relevant to this proposal:

- translating the BYM2 parameterization into executable PyMC code,
- fitting a minimal BYM2-style model on synthetic data,
- comparing it against a baseline iid random-effect model,
- validating that the spatial component captures neighborhood structure.

## Purpose of this notebook

The goal of this prototype is to validate the core BYM2 decomposition in a controlled setting. 
It demonstrates how structured (ICAR-based) and unstructured components can be combined in PyMC, 
and how such a model behaves on synthetic areal count data.

The notebook is intentionally lightweight and fast to run, while still covering simulation, 
model construction, inference, and basic validation.

## Mathematical form (simplified BYM2)

$$
\eta_i = \alpha + \sigma\left(\sqrt{1 - \rho}\,\theta_i^* + \sqrt{\rho}\,\phi_i^*\right)
$$

where:
- $\theta_i^*$ is an iid Normal component,
- $\phi_i^*$ is a structured ICAR component,
- $\sigma$ controls overall scale,
- $\rho$ controls structured vs unstructured mixing.

In [ ]:

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

az.style.use('arviz-darkgrid')
RNG = np.random.default_rng(2026)
np.set_printoptions(precision=3, suppress=True)


## 1. Construct a small adjacency graph

We use a small 4×4 lattice graph to keep the prototype computationally lightweight 
while still capturing meaningful spatial structure.

In [ ]:

def make_grid_adjacency(n_rows=4, n_cols=4):
    n = n_rows * n_cols
    W = np.zeros((n, n), dtype=int)

    def idx(r, c):
        return r * n_cols + c

    for r in range(n_rows):
        for c in range(n_cols):
            i = idx(r, c)
            for dr, dc in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                rr, cc = r + dr, c + dc
                if 0 <= rr < n_rows and 0 <= cc < n_cols:
                    W[i, idx(rr, cc)] = 1
    return W


def icar_scale_from_adjacency(W):
    # Compute a simple ICAR scaling factor from the Laplacian pseudoinverse.
    # This follows the BYM2 intuition that the structured component should be
    # scaled so its marginal variance is approximately 1 on average, making
    # sigma easier to interpret as the overall scale.
    D = np.diag(W.sum(axis=1))
    L = D - W
    evals, evecs = np.linalg.eigh(L)
    pos = evals > 1e-10
    L_pinv = (evecs[:, pos] / evals[pos]) @ evecs[:, pos].T
    marginal_vars = np.diag(L_pinv)
    scale = float(np.exp(np.mean(np.log(marginal_vars))))
    return scale, evals, evecs, L


W = make_grid_adjacency(4, 4)
scale_factor, evals, evecs, L = icar_scale_from_adjacency(W)
n_areas = W.shape[0]
coords = {'area': np.arange(n_areas)}

print('Number of areas:', n_areas)
print('ICAR scaling factor:', round(scale_factor, 4))
print('Adjacency matrix shape:', W.shape)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(W, cmap='Blues')
axes[0].set_title('Adjacency matrix W')
axes[0].set_xlabel('area')
axes[0].set_ylabel('area')

axes[1].imshow(L, cmap='viridis')
axes[1].set_title('Graph Laplacian D - W')
axes[1].set_xlabel('area')
axes[1].set_ylabel('area')

plt.tight_layout()
plt.show()


## 2. Simulate synthetic spatial count data

We simulate count data using a Poisson likelihood with exposure term $E$, 
following a standard disease-mapping formulation:

$$
y_i \sim \text{Poisson}(E_i \cdot \exp(\eta_i))
$$

The latent linear predictor $\eta_i$ includes a spatial effect constructed 
from both structured and unstructured components.

In [ ]:

def sample_scaled_icar(evals, evecs, scale, rng):
    pos = evals > 1e-10
    z = rng.normal(size=pos.sum())
    phi = evecs[:, pos] @ (z / np.sqrt(evals[pos]))
    phi = phi - phi.mean()  # explicit zero-sum centering
    phi = phi / np.sqrt(scale)
    return phi

alpha_true = -0.35
sigma_true = 0.60
rho_true = 0.70

E = RNG.integers(100, 151, size=n_areas)
theta_star_true = RNG.normal(size=n_areas)
phi_star_true = sample_scaled_icar(evals, evecs, scale_factor, RNG)
latent_true = sigma_true * (
    np.sqrt(1 - rho_true) * theta_star_true
    + np.sqrt(rho_true) * phi_star_true
)
mu_true = E * np.exp(alpha_true + latent_true)
y = RNG.poisson(mu_true)

sim_df = pd.DataFrame({
    'area': np.arange(n_areas),
    'E': E,
    'latent_true': latent_true,
    'mu_true': mu_true,
    'y': y,
})
sim_df.head()


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(phi_star_true.reshape(4, 4), cmap='coolwarm')
axes[0].set_title('True structured effect $\phi^*$')
axes[1].imshow(latent_true.reshape(4, 4), cmap='coolwarm')
axes[1].set_title('True latent BYM2 effect')
axes[2].imshow(y.reshape(4, 4), cmap='magma')
axes[2].set_title('Observed counts')
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

print(sim_df.describe().round(2))


## 3. Baseline model (iid random effects)

This baseline model ignores spatial dependence and includes only iid 
area-level variation. It serves as a comparison point to evaluate whether 
the spatial model provides additional structure.

In [ ]:

with pm.Model(coords=coords) as naive_model:
    E_data = pm.Data('E', E, dims='area')

    alpha = pm.Normal('alpha', 0, 2)
    sigma_u = pm.HalfNormal('sigma_u', 1)
    u_raw = pm.Normal('u_raw', 0, 1, dims='area')
    u = pm.Deterministic('u', sigma_u * u_raw, dims='area')

    eta = alpha + u + pm.math.log(E_data)
    pm.Poisson('y', mu=pm.math.exp(eta), observed=y, dims='area')

    idata_naive = pm.sample(
        draws=400,
        tune=400,
        chains=2,
        cores=1,
        target_accept=0.90,
        random_seed=2026,
        progressbar=False,
    )

az.summary(idata_naive, var_names=['alpha', 'sigma_u'], round_to=2)


## 4. BYM2-style spatial model

We now construct a BYM2-style model using existing PyMC components.

The spatial effect is defined as:

$$
\text{spatial}_i = \sigma\left(\sqrt{1 - \rho}\,\theta_i^* + \sqrt{\rho}\,\phi_i^*\right)
$$

where:
- $\theta_i^*$ is an iid Normal term,
- $\phi_i^*$ is modeled using an ICAR prior,
- $\sigma$ controls the overall scale,
- $\rho$ controls the mixing between structured and unstructured variation.

This construction reflects the BYM2 decomposition into structured and 
unstructured components using existing PyMC primitives.

In [ ]:

with pm.Model(coords=coords) as bym2_model:
    E_data = pm.Data('E', E, dims='area')

    alpha = pm.Normal('alpha', 0, 2)
    sigma = pm.HalfNormal('sigma', 1)
    rho = pm.Beta('rho', 0.5, 0.5)

    theta_star = pm.Normal('theta_star', 0, 1, dims='area')
    phi_raw = pm.ICAR('phi_raw', W=W, dims='area')
    phi_star = pm.Deterministic('phi_star', phi_raw / np.sqrt(scale_factor), dims='area')

    spatial = pm.Deterministic(
        'spatial',
        sigma * (
            pm.math.sqrt(1 - rho) * theta_star
            + pm.math.sqrt(rho) * phi_star
        ),
        dims='area',
    )

    eta = alpha + spatial + pm.math.log(E_data)
    pm.Poisson('y', mu=pm.math.exp(eta), observed=y, dims='area')

    idata_bym2 = pm.sample(
        draws=400,
        tune=400,
        chains=2,
        cores=1,
        target_accept=0.95,
        random_seed=2027,
        progressbar=False,
    )

az.summary(idata_bym2, var_names=['alpha', 'sigma', 'rho'], round_to=2)


## 5. Model comparison

A simple evaluation approach is to compare the baseline iid model with the 
BYM2-style spatial model in terms of fit and inferred spatial structure.

We expect the spatial model to better capture neighborhood dependence 
present in the simulated data.

In [ ]:

naive_latent_mean = idata_naive.posterior['u'].mean(('chain', 'draw')).values
bym2_latent_mean = idata_bym2.posterior['spatial'].mean(('chain', 'draw')).values

naive_rmse = float(np.sqrt(np.mean((naive_latent_mean - latent_true) ** 2)))
bym2_rmse = float(np.sqrt(np.mean((bym2_latent_mean - latent_true) ** 2)))

comparison = pd.DataFrame({
    'model': ['naive_iid', 'minimal_bym2'],
    'latent_rmse': [naive_rmse, bym2_rmse],
    'posterior_alpha_mean': [
        float(idata_naive.posterior['alpha'].mean()),
        float(idata_bym2.posterior['alpha'].mean()),
    ],
    'posterior_scale_mean': [
        float(idata_naive.posterior['sigma_u'].mean()),
        float(idata_bym2.posterior['sigma'].mean()),
    ],
    'posterior_rho_mean': [np.nan, float(idata_bym2.posterior['rho'].mean())],
})
comparison.round(3)


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(latent_true.reshape(4, 4), cmap='coolwarm')
axes[0].set_title('True latent effect')
axes[1].imshow(naive_latent_mean.reshape(4, 4), cmap='coolwarm')
axes[1].set_title(f'Naive posterior mean\nRMSE = {naive_rmse:.3f}')
axes[2].imshow(bym2_latent_mean.reshape(4, 4), cmap='coolwarm')
axes[2].set_title(f'Minimal BYM2 posterior mean\nRMSE = {bym2_rmse:.3f}')
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()


In [ ]:

summary_df = az.summary(idata_bym2, var_names=['alpha', 'sigma', 'rho'], round_to=2)
summary_df['true_value'] = [alpha_true, sigma_true, rho_true]
summary_df


## Key Takeaways

- Demonstrates understanding of the BYM2 decomposition
- Shows how ICAR-based spatial structure can be implemented in PyMC
- Provides a working prototype combining structured and unstructured effects
- Illustrates how spatial models differ from iid baselines
- Validates that inference runs successfully on a minimal example

## Limitations and Future Work

This prototype is not intended as a production-ready implementation. 
A full PyMC contribution would require:

- adjacency validation and preprocessing utilities,
- robust handling of dims / coords,
- extensive unit and integration tests,
- numerical validation across graph structures,
- documentation-quality examples,
- iterative refinement through maintainer review.

These aspects are explicitly addressed in the project plan of the proposal.